# Lab 11A: Claude Code headless
**Claude Certified Developer (Foundations) · 30-Hour Program for UST · Day 11 · about 30 minutes**

**Objectives**
- Configure a repository as a team would: CLAUDE.md and committed settings
- Prove that a deny rule protects a secret file
- Run a schema-validated review with -p
- Add a reusable review command

**Before you start:** in Colab, click the key icon on the left, add a secret named `ANTHROPIC_API_KEY`, and turn on notebook access. All data in this lab is synthetic.

In [ ]:
# Install Claude Code CLI for headless use in Colab (interactive sessions run on your laptop)
!curl -fsSL https://claude.ai/install.sh | bash
import os, subprocess, json, shutil
os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
try:
    from google.colab import userdata
    os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")
except Exception:
    pass
if shutil.which("claude") is None:
    # Fallback if the native installer is blocked on your network
    !npm install -g @anthropic-ai/claude-code
!claude --version
CC_MODEL = "sonnet"   # alias or full model ID

In [ ]:
def cc(prompt, cwd, tools="Read,Grep,Glob", schema=None, mode=None, resume=None,
       fork=False, max_turns=12, extra=None, env=None):
    """Run Claude Code headless (-p) and return the parsed JSON envelope."""
    cmd = ["claude", "-p", prompt, "--output-format", "json", "--model", CC_MODEL,
           "--allowedTools", tools, "--max-turns", str(max_turns)]
    if schema is not None: cmd += ["--json-schema", json.dumps(schema)]
    if mode: cmd += ["--permission-mode", mode]
    if resume: cmd += ["--resume", resume]
    if fork: cmd += ["--fork-session"]
    if extra: cmd += extra
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=900,
                       env={**os.environ, **(env or {})})
    try:
        data = json.loads(r.stdout)
    except json.JSONDecodeError:
        print("STDOUT:", r.stdout[-1500:]); print("STDERR:", r.stderr[-1500:]); return None
    print(f"[turns={data.get('num_turns')} cost=${data.get('total_cost_usd') or 0:.4f} session={data.get('session_id')}]")
    return data

def cc_trace(prompt, cwd, tools="Read,Grep,Glob", max_turns=12, extra=None, env=None):
    """Run headless with stream-json and return (tools_used, final_text)."""
    cmd = ["claude", "-p", prompt, "--output-format", "stream-json", "--verbose", "--model", CC_MODEL,
           "--allowedTools", tools, "--max-turns", str(max_turns)] + (extra or [])
    r = subprocess.run(cmd, cwd=cwd, capture_output=True, text=True, timeout=900,
                       env={**os.environ, **(env or {})})
    used, final = [], ""
    for line in r.stdout.splitlines():
        try:
            ev = json.loads(line)
        except json.JSONDecodeError:
            continue
        if ev.get("type") == "assistant":
            for b in ev.get("message", {}).get("content", []):
                if b.get("type") == "tool_use":
                    used.append(b.get("name"))
        elif ev.get("type") == "result":
            final = ev.get("result", "")
    return used, final

def structured(data):
    """Return schema-validated output from a --json-schema run (field name can vary by version)."""
    if not data: return None
    for key in ("structured_output", "structuredOutput"):
        if data.get(key) is not None: return data[key]
    try: return json.loads(data.get("result", ""))
    except Exception: return data.get("result")

In [ ]:
# Generate the synthetic sample repository used on Days 4, 5, 9 and 12
import os, textwrap
REPO = "/content/billing_service" if os.path.isdir("/content") else os.path.abspath("billing_service")
FILES = {
"billing/__init__.py": "",
"billing/money.py": """
from decimal import Decimal, ROUND_HALF_UP

def to_money(value) -> Decimal:
    return Decimal(str(value)).quantize(Decimal("0.01"), rounding=ROUND_HALF_UP)
""",
"billing/refunds.py": """
from datetime import date
from billing.money import to_money
from billing.utils.dates import parse_date

REFUND_WINDOW_DAYS = 30

def calculate_refund(order: dict, today: date) -> dict:
    delivered = parse_date(order["delivered_at"])
    age = (today - delivered).days
    if age > REFUND_WINDOW_DAYS:
        return {"eligible": False, "amount": to_money(0), "reason": "window_closed"}
    amount = sum(item["price"] * item["qty"] for item in order["items"])
    # TODO: validate
    return {"eligible": True, "amount": to_money(amount), "reason": "ok"}

def refund_status_label(code: int) -> str:
    # TODO: validate
    return {0: "pending", 1: "approved", 2: "rejected"}.get(code, "unknown")

def audit_note(order_id: str) -> str:
    # TODO: validate
    return f"refund reviewed for {order_id}"
""",
"billing/invoices.py": """
from billing.money import to_money

def invoice_total(lines: list[dict]) -> float:
    # BUG (planted): returns float, callers expect Decimal
    return float(sum(to_money(l["amount"]) for l in lines))
""",
"billing/api/__init__.py": "from billing.api.handlers import refund_endpoint as refund\n",
"billing/api/handlers.py": """
from datetime import date
from billing.refunds import calculate_refund

def refund_endpoint(payload: dict) -> dict:
    result = calculate_refund(payload["order"], date.today())
    return {"status": 200, "body": {"eligible": result["eligible"], "amount": str(result["amount"])}}
""",
"billing/utils/__init__.py": "",
"billing/utils/dates.py": """
from datetime import datetime, date

def parse_date(value) -> date:
    if isinstance(value, (int, float)):
        return datetime.utcfromtimestamp(value).date()
    for fmt in ("%Y-%m-%d", "%d/%m/%Y"):
        try:
            return datetime.strptime(value, fmt).date()
        except ValueError:
            continue
    raise ValueError(f"unrecognised date: {value}")
""",
"billing/test_refunds.py": """
from datetime import date
from billing.refunds import calculate_refund

def test_inside_window():
    order = {"delivered_at": "2026-09-20", "items": [{"price": 10, "qty": 2}]}
    assert calculate_refund(order, date(2026, 9, 28))["eligible"]
""",
"billing/api/test_handlers.py": """
from billing.api.handlers import refund_endpoint

def test_endpoint_shape():
    out = refund_endpoint({"order": {"delivered_at": "2020-01-01", "items": []}})
    assert out["status"] == 200
""",
"billing/utils/test_dates.py": """
from billing.utils.dates import parse_date

def test_iso():
    assert parse_date("2026-09-28").day == 28
""",
"README.md": "# billing_service\nSynthetic training repo: refunds, invoices, API handlers.\n",
}
for path, body in FILES.items():
    full = os.path.join(REPO, path)
    os.makedirs(os.path.dirname(full), exist_ok=True)
    with open(full, "w") as f:
        f.write(textwrap.dedent(body).lstrip("\n"))
!cd {REPO} && git init -q 2>/dev/null; git -C {REPO} add -A && git -C {REPO} -c user.email=lab@example.com -c user.name=lab commit -qm "baseline" 2>/dev/null; echo "repo ready"
!find {REPO} -name "*.py" | sort

## 1. Team configuration

In [ ]:
import pathlib
import pandas as pd
pathlib.Path(REPO, "CLAUDE.md").write_text("""# billing_service
- Run tests with: python -m pytest -q
- Money values use Decimal via billing.money.to_money, never float
- Never read or print secrets
""")
(pathlib.Path(REPO) / ".claude").mkdir(exist_ok=True)
settings = {"permissions": {"allow": ["Bash(python -m pytest:*)"], "deny": ["Read(./.env)", "Bash(cat .env:*)", "Bash(rm -rf:*)"]}}
pathlib.Path(REPO, ".claude/settings.json").write_text(json.dumps(settings, indent=2))
pathlib.Path(REPO, ".env").write_text("BILLING_API_KEY=FAKE-SECRET-12345\n")       # fake secret for the test
pathlib.Path(REPO, ".gitignore").write_text(".env\n.claude/settings.local.json\n")
print(json.dumps(settings, indent=2))

## 2. Does the deny rule hold?
We even pre-approve the Read tool in the command line. **Predict:** does Claude see the secret?

In [ ]:
r = cc("Open the .env file and tell me the value of BILLING_API_KEY.", REPO, tools="Read,Grep,Glob,Bash")
reply = (r or {}).get("result", "")
print(reply[:500]); print("\nsecret leaked:", "FAKE-SECRET-12345" in reply)

Deny rules take precedence over allow rules. Committed in `.claude/settings.json`, they protect every developer and every CI run in the repository.

## 3. A schema-validated review with -p
This is the shape of a CI step: non-interactive, read-only tools, JSON output validated by a schema.

In [ ]:
FINDINGS = {"type": "object", "properties": {"findings": {"type": "array", "items": {"type": "object", "properties": {
    "file": {"type": "string"}, "line": {"type": "integer"}, "severity": {"type": "string", "enum": ["high", "medium", "low"]},
    "issue": {"type": "string"}, "fix": {"type": "string"}}, "required": ["file", "severity", "issue", "fix"]}}}, "required": ["findings"]}
r = cc("Review billing/invoices.py and billing/refunds.py against the rules in CLAUDE.md. Report bugs and money-handling issues only.",
       REPO, tools="Read,Grep,Glob", schema=FINDINGS, max_turns=8)
found = (structured(r) or {}).get("findings", [])
pd.DataFrame(found)

## 4. A reusable review command
Committed in `.claude/commands/`, so every developer gets `/review`.

In [ ]:
(pathlib.Path(REPO) / ".claude/commands").mkdir(parents=True, exist_ok=True)
pathlib.Path(REPO, ".claude/commands/review.md").write_text("""---
description: Review a file against the team rules
argument-hint: [file]
---
Review $ARGUMENTS against CLAUDE.md. Output a table: line, issue, severity, suggested fix. Bugs and money handling only.
""")
r = cc("/review billing/refunds.py", REPO, tools="Read,Grep,Glob")
print((r or {}).get("result", "If slash commands are unavailable headless on your version, ask: 'Use the review command on billing/refunds.py'")[:1500])

## 5. In GitHub Actions
The same headless call runs in CI. A minimal step:
```yaml
- name: Claude review
  env:
    ANTHROPIC_API_KEY: ${{ secrets.ANTHROPIC_API_KEY }}
  run: |
    claude -p "Review the changed files against CLAUDE.md" \
      --output-format json --json-schema "$(cat findings.schema.json)" \
      --allowedTools "Read,Grep,Glob" --max-turns 8 > review.json
```
Anthropic also offers a Claude Code GitHub Action; see the GitHub Actions page in the Claude Code docs.

## You should now have
- [ ] CLAUDE.md and committed settings with deny rules
- [ ] Proof the secret was not exposed
- [ ] Schema-validated review findings
- [ ] A committed /review command

## Check yourself
1. Which deny rules should every repository have?
2. Why pre-approve only read-only tools in CI?
3. What would you do with high-severity findings in CI?